## 1. Standalone Deployment Environment

This notebook is designed as an independent inference pipeline. It simulates the exact environment of an end-user (e.g., a clinician) downloading the repository from GitHub. 

By loading the pre-trained model with compile=False, we bypass the need to define the custom training loss functions and metrics. This heavily reduces library dependencies and ensures the tool is lightweight and ready for immediate deployment on new, unseen CT scans.

In [ ]:
import os
import numpy as np
import nibabel as nib
import tensorflow as tf
from pathlib import Path

# Suppress verbose TensorFlow hardware warnings for a cleaner output
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# --- 1. INITIALIZE THE ENVIRONMENT & LOAD MODEL ---
MODEL_PATH = "unet_sarcopenia_best.keras"

print(f"Loading pre-trained U-Net from: {MODEL_PATH}")
# compile=False allows loading for inference without custom loss function dependencies
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print("Model loaded successfully and ready for inference.")

Loading pre-trained U-Net from: unet_sarcopenia_best.keras
Model loaded successfully and ready for inference.


## 2. Integrated Clinical Assessment Pipeline

The following function acts as the core of the medical software. It executes three distinct operations in a single pass:
1. **AI Segmentation:** Reads a raw NIfTI .nii file, applies the required Hounsfield windowing, and generates a muscle mask using the U-Net.
2. **NIfTI Reconstruction:** Resizes the predicted mask back to its native spatial resolution and saves it, injecting the original affine matrix and header so it flawlessly overlays on the original scan in software like 3D Slicer.
3. **Clinical Sarcopenia Math:** Implements the clinical logic to calculate the cross-sectional area (cm²) and mean radiodensity (HU) directly from the native arrays, outputting an automated text report.

### Crucial Step: Nearest-Neighbor Resampling

When the U-Net generates its prediction, it outputs a mask at the standardized `256x256` resolution. However, the original clinical CT scan might be `512x512` (or another native resolution). To reconstruct the final NIfTI file and perform accurate clinical math, the mask is up-sampled back to its native shape.

**Why `method='nearest'`?**
Standard image resizing algorithms (like bilinear interpolation) average neighboring pixel values. If applied to a mask containing class `1` (Abdominals) and class `2` (Psoas), the math would blend them and create false decimal classes (e.g., `1.5`) along the borders, completely corrupting the segmentation. By strictly using **Nearest-Neighbor interpolation**, the algorithm chooses the exact integer value of the closest pixel. This preserves the absolute categorical integrity of the labels (`0`, `1`, or `2`) so the physical area calculation (cm²) remains mathematically perfect.

In [ ]:
def predict_save_and_report_sarcopenia(input_ct_path, output_mask_path, model, target_size=(256, 256)):
    
    # Executes U-Net prediction on a raw CT, saves the NIfTI mask, 
    # and generates a clinical sarcopenia assessment report.
    
    input_ct_path = Path(input_ct_path)
    output_mask_path = Path(output_mask_path)
    

    # 1. LOAD AND PREPROCESS DATA
 
    nii_img = nib.load(input_ct_path)
    original_data = nii_img.get_fdata()
    original_affine = nii_img.affine   
    original_header = nii_img.header   
    original_shape = original_data.shape

    # Neural Network Preprocessing (HU Windowing)
    img_np = np.squeeze(original_data)
    HU_MIN, HU_MAX = -30.0, 150.0
    img_np = np.clip(img_np, HU_MIN, HU_MAX)
    img_np = (img_np - HU_MIN) / (HU_MAX - HU_MIN)

    # Tensor conversion and spatial resizing
    img_tensor = tf.expand_dims(tf.convert_to_tensor(img_np, dtype=tf.float32), axis=-1)
    img_resized = tf.image.resize(img_tensor, target_size, method='bilinear').numpy()
    input_batch = np.expand_dims(img_resized, axis=0)

     
    # 2. U-NET PREDICTION & POST-PROCESSING
     
    preds = model.predict(input_batch, verbose=0)
    pred_mask_256 = np.argmax(preds, axis=-1)[0] 

    # Restore to native resolution using nearest neighbor to preserve discrete classes
    pred_mask_original_shape = tf.image.resize(
        np.expand_dims(pred_mask_256, axis=-1),
        original_shape[:2], 
        method='nearest'
    ).numpy().squeeze()

    # Restore the phantom Z-dimension if present in the original 2D slice (e.g., 512, 512, 1)
    if len(original_shape) == 3:
        pred_mask_original_shape = np.expand_dims(pred_mask_original_shape, axis=-1)

    # Save the new NIfTI mask
    new_nii = nib.Nifti1Image(pred_mask_original_shape.astype(np.int16), affine=original_affine, header=original_header)
    nib.save(new_nii, output_mask_path)
    print(f"[SUCCESS] NIfTI Mask saved to: {output_mask_path}")

     
    # 3. CLINICAL CALCULATIONS (Dr. Rizzetto's Logic)
     
    segmentation_data = pred_mask_original_shape
    ct_data = original_data # Use raw data for accurate HU density

    # Pixel counting
    area_1_2 = np.count_nonzero(np.logical_or(segmentation_data == 1, segmentation_data == 2))
    area_2 = np.count_nonzero(segmentation_data == 2)
    
    # Extract physical pixel spacing from native header
    x_pixel_spacing = original_header['pixdim'][1]
    y_pixel_spacing = original_header['pixdim'][2]
    
    # Calculate areas in cm²
    area_1_2_cm2 = (area_1_2 * x_pixel_spacing * y_pixel_spacing) / 100  
    area_2_cm2 = (area_2 * x_pixel_spacing * y_pixel_spacing) / 100  
    
    # Boolean masks for density calculation
    mask_1_2 = np.logical_or(segmentation_data == 1, segmentation_data == 2)
    mask_2 = segmentation_data == 2
    
    # Calculate mean density (HU)
    mean_density_1_2 = np.mean(ct_data[mask_1_2]) if np.any(mask_1_2) else 0
    mean_density_2 = np.mean(ct_data[mask_2]) if np.any(mask_2) else 0

    # 4. REPORT GENERATION
    patient_id = input_ct_path.name.replace('.nii', '').replace('.gz', '')
    
    report_text = f"""
CLINICAL SARCOPENIA ASSESSMENT REPORT (AI-Generated)

Patient ID: {patient_id}
Source File: {input_ct_path.name}
--------------------------------------------------
L3SMI (Total Muscle Body - Class 1+2):
  -> Area: {area_1_2_cm2:.1f} cm²
  -> Mean Density: {mean_density_1_2:.0f} HU

L3 Psoas Muscle (Class 2):
  -> Area: {area_2_cm2:.1f} cm²
  -> Mean Density: {mean_density_2:.0f} HU

L4 Psoas Muscle (Class 3):
  -> Area: {area_3_cm2:.1f} cm²
  -> Mean Density: {f"{mean_density_3:.0f} HU" if isinstance(mean_density_3, float) else str(mean_density_3)}
--------------------------------------------------
Report generated automatically via Custom U-Net Pipeline.
"""

    print("\n" + report_text)

    # Save text report to disk
    report_output_path = output_mask_path.with_suffix('').with_suffix('.txt')
    with open(report_output_path, 'w', encoding='utf-8') as f:
        f.write(report_text)
    print(f"[SUCCESS] Clinical Report saved to: {report_output_path}\n")

## 3. Execution Example

To test the pipeline, place any unlabelled .nii CT scan in the demo_data folder. The script will output both the predicted mask and the automated clinical report in the results folder.

In [ ]:
# Setup local directories for the GitHub repository structure
DEMO_FOLDER = Path(r"C:\Use\folder\that\will_contain\the\slices")
RESULTS_FOLDER = Path(r"C:\Use\Your\Folder\that\will_go\the_results")

DEMO_FOLDER.mkdir(exist_ok=True)
RESULTS_FOLDER.mkdir(exist_ok=True)

# Select a target file (ensure you place a sample file in demo_data before running)
# Example: We assume a file named "sample_patient.nii" exists in demo_data
target_ct_file = DEMO_FOLDER / "slice_subject_1.nii"

if target_ct_file.exists():
    output_mask = RESULTS_FOLDER / f"predicted_mask_{target_ct_file.name}"
    
    # Execute the clinical inference pipeline
    predict_save_and_report_sarcopenia(
        input_ct_path=target_ct_file, 
        output_mask_path=output_mask, 
        model=model
    )
else:
    print(f"Please place a valid .nii file at {target_ct_file} to run the inference test.")

 Maschera NIfTI salvata in: C:\Users\vella\Desktop\tirocinio\Dataset\predizioni_modello\predicted_mask_slice_subject_1.nii

Patient ID: slice_subject_1
Source File: slice_subject_1.nii
--------------------------------------------------
L3SMI (Total Muscle Body - Class 1+2):
  -> Area: 96.1 cm²
  -> Mean Density: 12 HU

L3 Psoas Muscle (Class 2):
  -> Area: 10.4 cm²
  -> Mean Density: 27 HU

L4 Psoas Muscle (Class 3):
  -> Area: 0.0 cm²
  -> Mean Density: N/A
--------------------------------------------------
Report generated automatically via Custom U-Net Pipeline.

 Report clinico salvato in: C:\Users\vella\Desktop\tirocinio\Dataset\predizioni_modello\predicted_mask_slice_subject_1_report.txt



## Pipeline Architecture: Clinical Inference and Reporting

This flowchart illustrates the standalone deployment architecture. Unlike the training pipeline, this environment is fully decoupled from custom metric dependencies, prioritizing speed, minimal library requirements, and direct clinical utility.

```text

                 PHASE 1: INITIALIZATION & INGESTION

[ Pre-Trained Weights (.keras) ]        [ Unseen Raw CT Scan (.nii) ]
               │                                      │
               ├──> Load Model (compile=False)        │
               │                                      │
               ▼                                      ▼
[ Lightweight Inference Engine ]        [ Metadata Extraction (Affine/Header) ]


                 PHASE 2: PREPROCESSING & AI PREDICTION

                                                      │
                                                      ├──> Hounsfield Windowing [-30, +150]
                                                      ├──> Min-Max Normalization [0, 1]
                                                      ├──> Spatial Downsampling (256x256)
                                                      │
                                                      ▼
                                           [ U-Net Neural Network ]
                                                      │
                                                      ▼
                                        [ Predicted Categorical Mask ]


                 PHASE 3: PHYSICAL RECONSTRUCTION & CLINICAL MATH

                                                      │
                                                      ├──> Restore Native Resolution (Nearest)
                                                      ├──> Re-inject Original NIfTI Metadata
                                                      ├──> Compute Physical Area (cm²) via Voxel Spacing
                                                      └──> Compute Mean Radiodensity (HU) from Raw CT
                                                      │
                                                      ▼

                 PHASE 4: OUTPUT GENERATION

                                                      │
                                                      ├──> Export Compatible NIfTI Mask
                                                      └──> Export Formatted Sarcopenia Report
                                                      │
                                                      ▼
[ Final Clinical Outputs: predicted_mask.nii  &  predicted_mask_report.txt ]
```